# Puzzle

https://thefiddler.substack.com/p/can-you-number-the-number-cubes

### This Week’s Fiddler

From John Torrey comes a puzzle any sufficiently clever Bond villain would appreciate:

You have four number cubes, where each face of each cube can display one numeric digit from 0 to 9. You can make various numbers by picking three faces on three distinct cubes and lining them up. For example, you can make the number “123” by lining up three cubes to show 1-2-3, and you can make the number “7” by lining up three cubes to show 0-0-7.

Importantly, any face with a “6” can also be used to display a “9” by flipping the cube around, and vice versa. However, no other pairs of digits are interchangeable in this way.

You can choose which digits to place on the various faces of the four cubes. Your goal is to be able to make all the whole numbers from 1 to N, without skipping any numbers in between. With optimal design, what is the greatest possible value of N?

### This Week’s Extra Credit

From John Torrey also comes some Extra Credit:

How many distinct ways can you assign numbers to the four cubes that result in this greatest possible value of N?

Note that the cubes are not ordered in any way. And, very importantly, swapping digits between two faces on a single cube does not count as producing a distinct arrangement. In other words, don’t worry about the various ways to assign six given digits to the faces of a cube. (Also, “6” and “9” count as the same digit here.")

# Solution

I was trying to work this out by hand, but there are a lot of cases, and it is hard to be sure you didn't miss something. So, I am going to fall back to some exhaustive coding and see how it goes.

There is no value to having the same digit twice on a single cube, so I think it could be safe to assume that the digits are all distinct. But I am not totally sure. So, maybe try both ways.

In [1]:
all_digits = [0,1,2,3,4,5,6,7,8]
set_all_digits = set(all_digits)

# Ignore digit 9, as it can be represented as 6 rotated.
possible_cubes = []
distinct_digits_cubes = []
for a in range(0,9):
    for b in range(a,9):
        for c in range(b,9):
            for d in range(c,9):
                for e in range(d,9):
                    for f in range(e,9):
                        possible_cubes.append((a,b,c,d,e,f))
                        if a != b and b != c and c != d and d != e and e != f:
                            # Since the digits are sorted, we can just check adjacent digits for distinctness.
                            distinct_digits_cubes.append((a,b,c,d,e,f))

print (len(possible_cubes))
print (len(distinct_digits_cubes))

3003
84


In [2]:
import itertools
def cubesets(cubes):
    for cset in itertools.combinations_with_replacement(cubes, 4):
        yield cset

In [3]:
from functools import cache
@cache
def number_to_digits(n):
    digits = []
    for i in range(3):
        digit = n % 10
        n //= 10
        if digit == 9:
            digit = 6
        digits.append(digit)
    digits.reverse()
    return digits

In [4]:
def digits_can_be_covered_by_cubeset(digits, cubeset):
    # We can just check all 24 permutations of the cubes, since there are only 4 cubes and 3 digits.
    for cube_perm in itertools.permutations(cubeset, 3):
        if digits[0] in cube_perm[0] and digits[1] in cube_perm[1] and digits[2] in cube_perm[2]:
            return True
    return False

In [5]:
def find_range_of_cubeset(cubeset):
    for i in range(1,1000):
        digits = number_to_digits(i)
        if not (digits[0] <= digits[1] <= digits[2]):
            # This number has digits that are not in non-decreasing order, so we can skip it because it is a rearrangement of a smaller number that we will have already checked.
            continue
        if not digits_can_be_covered_by_cubeset(digits, cubeset):
            return i-1
    return 999

In [6]:

def survey_cubesets(cubes, cover_filter=False, print_range=None):
    range_counts = {}
    if print_range is not None:
        print(f"Cubesets with Range {print_range}:")
    for cset in cubesets(cubes):
        if cover_filter and set(cset[0]) | set(cset[1]) | set(cset[2]) | set(cset[3]) != set_all_digits:
            # This cubeset doesn't even cover all the digits, so we can skip it.
            continue
        r = find_range_of_cubeset(cset)
        if r not in range_counts:
            range_counts[r] = 0
        range_counts[r] += 1
        if r == print_range:
            print(f"  {range_counts[r]}: {cset}")
    print("Range: count of ways to achieve that range")        
    total_count = 0
    for r in sorted(range_counts.keys()):
        c = range_counts[r]
        print (f"  {r}: {c}")
        total_count += c
    print(f"Total: {total_count}")

In [7]:
survey_cubesets(distinct_digits_cubes, cover_filter=True, print_range=776)

Cubesets with Range 776:
  1: ((0, 1, 2, 3, 4, 5), (0, 1, 2, 3, 4, 6), (1, 2, 5, 6, 7, 8), (3, 4, 5, 6, 7, 8))
  2: ((0, 1, 2, 3, 4, 5), (0, 1, 2, 3, 4, 6), (1, 3, 5, 6, 7, 8), (2, 4, 5, 6, 7, 8))
  3: ((0, 1, 2, 3, 4, 5), (0, 1, 2, 3, 4, 6), (1, 4, 5, 6, 7, 8), (2, 3, 5, 6, 7, 8))
  4: ((0, 1, 2, 3, 4, 5), (0, 1, 2, 3, 5, 6), (1, 2, 4, 6, 7, 8), (3, 4, 5, 6, 7, 8))
  5: ((0, 1, 2, 3, 4, 5), (0, 1, 2, 3, 5, 6), (1, 3, 4, 6, 7, 8), (2, 4, 5, 6, 7, 8))
  6: ((0, 1, 2, 3, 4, 5), (0, 1, 2, 3, 5, 6), (1, 4, 5, 6, 7, 8), (2, 3, 4, 6, 7, 8))
  7: ((0, 1, 2, 3, 4, 5), (0, 1, 2, 3, 6, 7), (1, 2, 4, 5, 6, 8), (3, 4, 5, 6, 7, 8))
  8: ((0, 1, 2, 3, 4, 5), (0, 1, 2, 3, 6, 7), (1, 3, 4, 5, 6, 8), (2, 4, 5, 6, 7, 8))
  9: ((0, 1, 2, 3, 4, 5), (0, 1, 2, 3, 6, 7), (1, 4, 5, 6, 7, 8), (2, 3, 4, 5, 6, 8))
  10: ((0, 1, 2, 3, 4, 5), (0, 1, 2, 3, 6, 8), (1, 2, 4, 5, 6, 7), (3, 4, 5, 6, 7, 8))
  11: ((0, 1, 2, 3, 4, 5), (0, 1, 2, 3, 6, 8), (1, 3, 4, 5, 6, 7), (2, 4, 5, 6, 7, 8))
  12: ((0, 1, 2, 3, 4, 5), 

In [8]:
# survey_cubesets(possible_cubes, cover_filter=True, print_range=776)
# Could not complete in 2+ hours. Giving up.

# Conclusion

Looks like 776 is the max possible N, and there are 855 ways to create cubes that yield this range of N.

I did make the assumption that in any cube all the digits are distinct. Originally, I was not sure if there is some way to create a cubeset with repeated digits in a single cube that still hits the max range.
But now I realized that for a range of 776, we need 3 copies of digits 1,2,3,4,5,6 (for 111, 222, etc) and we need 2 copies of digits 0, 7, 8 (for 001, 077, 088, etc). That's 18 + 6 = 24 digits. And all copies of each digit must be on different cubes. So, there's no wiggle room. So, in fact, any cubeset with repeated digits on a single cube must have a smaller range.

Does this give us a way to count the number of ways directly? It probably does, but I am not seeing it too obviously yet.

UPDATE: Originally, I had a bug in the cubsets function. It was using combinations instead of combinations_with_replacement. Fortunately, the answer of 855 is the same both ways. This means that none of the 855 cubesets has 2 identical cubes. That sounds plausible, but it's also not immediately obvious that it must have been so.